# Evaluate Performance of Qwen3-8B-AWQ

In [1]:
import tqdm
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from functools import partial
import gc
import os
os.environ['https_proxy'] = 'http://192.168.1.12:7891'

debug = True

if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

/root/workspace/qwen_cpu_deployment/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here we use wikitext-2 dataset for perplexity evaluation. The dataset is automatically downloaded by the code.

In [2]:
print("Loding wikitext datasets ...")
testenc = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test', cache_dir="~/.cache/huggingface/datasets")
print("Done")

Loding wikitext datasets ...
Done


In [3]:
def evaluate(model, testenc, tokenizer):
    # we control the text length to avoid error posed by tiktoken
    testenc = tokenizer("\n\n".join(testenc['text']), return_tensors='pt')
    testenc = testenc.input_ids.to(model.device)
    nsamples = 40
    model = model.eval()

    nlls = []
    for i in tqdm.tqdm(range(nsamples), desc="evaluating Qwen on wikitext"):
        batch = testenc[:, (i * 1024):((i + 1) * 1024)].to(model.device)
        with torch.no_grad():
            lm_logits = model(batch).logits
        shift_logits = lm_logits[:, :-1, :].contiguous().float()
        shift_labels = testenc[:, (i * 1024):((i + 1) * 1024)][:, 1:]
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        neg_log_likelihood = loss.float() * 1024
        nlls.append(neg_log_likelihood)

    return torch.exp(torch.stack(nlls).sum() / (nsamples * 1024))

def get_model_size(model: nn.Module, data_width=16, group_size=-1):
    # store the quantization parameters: 1 fp16 scaling factors and 1 int4 zero point
    # this might not be precise
    if group_size != -1:
        data_width += (16 + 4) / group_size
    num_elements = 0
    for param in model.parameters():
        num_elements += param.numel()
    return num_elements * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

# Evaluate the performance of FP16 Qwen
## PPL

In [4]:

model_name = "Qwen/Qwen3-8B-AWQ"

In [5]:

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)

/root/workspace/qwen_cpu_deployment/.venv/lib/python3.10/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.49it/s]


In [6]:
fp32_perplexity = evaluate(model, testenc, tokenizer)
print(f"\nmodel perplexity: {fp32_perplexity:.2f}")


Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 131072). Running this sequence through the model will result in indexing errors
evaluating Qwen on wikitext: 100%|██████████| 40/40 [00:05<00:00,  6.85it/s]



model perplexity: 11.52


In [7]:
del model
gc.collect()
torch.cuda.empty_cache()

## GSM8k

In [8]:
!lm-eval --tasks gsm8k --model vllm --model_args pretrained=Qwen/Qwen3-8B-AWQ,max_model_len=8192,dtype=float16 --batch_size auto --trust_remote_code

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 05-30 13:57:21 [__init__.py:243] Automatically detected platform cuda.
2025-05-30:13:57:24 INFO     [__main__:428] Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`
2025-05-30:13:57:24 INFO     [__main__:440] Selected Tasks: ['gsm8k']
2025-05-30:13:57:24 INFO     [evaluator:185] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-05-30:13:57:24 INFO     [evaluator:223] Initializing vllm model, with arguments: {'pretrained': 'Qwen/Qwen3-8B-AWQ', 'max_model_len': 8192, 'dtype': 'float16', 'trust_remote_code': True}
INFO 05-30 13:57:24 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 05-30 13:57:24 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 05-30 13:57:24 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plu